In [1]:
import numpy as np
import pandas as pd

In [2]:
df=pd.read_csv('diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
# find correlation 
df.corr()['Outcome']

Pregnancies                 0.221898
Glucose                     0.466581
BloodPressure               0.065068
SkinThickness               0.074752
Insulin                     0.130548
BMI                         0.292695
DiabetesPedigreeFunction    0.173844
Age                         0.238356
Outcome                     1.000000
Name: Outcome, dtype: float64

In [4]:
# create x and y
x=df.iloc[:,:-1].values
y=df.iloc[:,-1].values

In [5]:
# Scaling 
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

x=scaler.fit_transform(x)
x

array([[ 0.63994726,  0.84832379,  0.14964075, ...,  0.20401277,
         0.46849198,  1.4259954 ],
       [-0.84488505, -1.12339636, -0.16054575, ..., -0.68442195,
        -0.36506078, -0.19067191],
       [ 1.23388019,  1.94372388, -0.26394125, ..., -1.10325546,
         0.60439732, -0.10558415],
       ...,
       [ 0.3429808 ,  0.00330087,  0.14964075, ..., -0.73518964,
        -0.68519336, -0.27575966],
       [-0.84488505,  0.1597866 , -0.47073225, ..., -0.24020459,
        -0.37110101,  1.17073215],
       [-0.84488505, -0.8730192 ,  0.04624525, ..., -0.20212881,
        -0.47378505, -0.87137393]], shape=(768, 8))

In [6]:
# now create train test split 
from sklearn.model_selection import train_test_split

xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.2,random_state=1)

xtrain.shape,xtest.shape

((614, 8), (154, 8))

In [7]:
# creating model
import tensorflow
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense,Dropout

model=Sequential()
model.add(Dense(32,activation='relu',input_dim=8))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

model.fit(xtrain,ytrain,batch_size=32,epochs=100,validation_data=(xtest,ytest))

c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6498 - loss: 0.6681 - val_accuracy: 0.6494 - val_loss: 0.6462
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6564 - loss: 0.6193 - val_accuracy: 0.6623 - val_loss: 0.6078
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6710 - loss: 0.5845 - val_accuracy: 0.6948 - val_loss: 0.5814
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6889 - loss: 0.5589 - val_accuracy: 0.7078 - val_loss: 0.5596
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7296 - loss: 0.5379 - val_accuracy: 0.6948 - val_loss: 0.5441
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7459 - loss: 0.5222 - val_accuracy: 0.7338 - val_loss: 0.5312
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7590 - loss: 0.5100 - val_accuracy: 0.7727 - val_loss: 0.5196
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7704 - loss: 0.4996 - val_accuracy: 0.7857 - 

# Hyperparameter Tuning
- The problem is when we create model we don't now which parameter is best or which give best accuracy ,so to find best parameter we need to try with every parameter and then find out which is best, so this is done by hyperparameter tunning so here we try to solve following condition:
1. select appropiate optimizer
2. no of node in a layer
3. select no of layer
4. all in one model

## 1. select appropiate optimizer 

In [8]:
# 1. select appropiate optimizer 
import kerastuner as kt

def build_model(hp):
    model=Sequential()
    model.add(Dense(32,activation='relu',input_dim=8))
    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer=hp.Choice('optimizer',values=['adam','sgd','rmsprop','adadelta']),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    
    return model

# create tunner object
tunner=kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5)



Reloading Tuner from .\untitled_project\tuner0.json


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5192\2661016720.py:2: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt


In [9]:
# find best hyperparameter
tunner.search(xtrain,ytrain,epochs=5,validation_data=(xtest,ytest))

# get best parameter
print(tunner.get_best_hyperparameters()[0].values)

# get best model
model=tunner.get_best_models(num_models=1)[0]
model.summary()

{'optimizer': 'adam'}



c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

## 2. select no of node in a layer

In [10]:
def build_model(hp):
    model=Sequential()
    units=hp.Int('unit',min_value=8,max_value=128)

    model.add(Dense(units=units,activation='relu',input_dim=8))
    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])
    return model

# create object
tunner=kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5,directory='Mydir',project_name='Mybuddy')

tunner.search(xtrain,ytrain,epochs=5,validation_data=(xtest,ytest))

print(tunner.get_best_hyperparameters()[0].values)

# best model extract
model=tunner.get_best_models(num_models=1)[0]

model.fit(xtrain,ytrain,epochs=100,initial_epoch=5,validation_data=(xtest,ytest))

Reloading Tuner from Mydir\Mybuddy\tuner0.json
{'unit': 94}
Epoch 6/100


c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7671 - loss: 0.5035 - val_accuracy: 0.7792 - val_loss: 0.4961
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7671 - loss: 0.4826 - val_accuracy: 0.7857 - val_loss: 0.4867
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7720 - loss: 0.4733 - val_accuracy: 0.7857 - val_loss: 0.4790
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7736 - loss: 0.4665 - val_accuracy: 0.7857 - val_loss: 0.4765
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7736 - loss: 0.4615 - val_accuracy: 0.7792 - val_loss: 0.4752
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7752 - loss: 0.4578 - val_accuracy: 0.7792 - val_loss: 0.4736
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7785 - loss: 0.4550 - val_accuracy: 0.7727 - val_loss: 0.4743
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7801 - loss: 0.4533 - val_accuracy: 0.7792 - val_loss

# 3. no of layers

In [19]:
def build_model3(hp):
    model=Sequential()
    model.add(Dense(72,activation='relu',input_dim=8))

    for i in range(hp.Int('num_layer',min_value=1,max_value=10)):
        model.add(Dense(72,activation='relu'))
    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])
    return model

tunner3=kt.RandomSearch(build_model3,objective='val_accuracy',max_trials=5,directory='Mydir',project_name='nooflayer')
tunner3.search(xtrain,ytrain,epochs=5,validation_data=(xtest,ytest))


Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.8051947951316833

Best val_accuracy So Far: 0.8051947951316833
Total elapsed time: 00h 00m 20s


In [20]:
tunner3.get_best_hyperparameters()[0].values

model=tunner3.get_best_models(num_models=1)[0]
model.fit(xtrain,ytrain,epochs=100,initial_epoch=5,validation_data=(xtest,ytest))

Epoch 6/100


c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 12 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.7752 - loss: 0.4517 - val_accuracy: 0.7987 - val_loss: 0.4689
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7834 - loss: 0.4397 - val_accuracy: 0.7922 - val_loss: 0.4648
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7850 - loss: 0.4328 - val_accuracy: 0.7727 - val_loss: 0.4641
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7834 - loss: 0.4251 - val_accuracy: 0.7857 - val_loss: 0.4670
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8013 - loss: 0.4158 - val_accuracy: 0.7662 - val_loss: 0.5053
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7948 - loss: 0.4082 - val_accuracy: 0.7922 - val_loss: 0.4778
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8094 - loss: 0.3989 - val_accuracy: 0.7662 - val_loss: 0.5237
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8225 - loss: 0.3873 - val_accuracy: 0.7597 - val_los

# 4 all in one model

In [28]:
def build_model4(hp):
    model=Sequential()
    counter=0
    
    for i in range(hp.Int('num_layer',min_value=1,max_value=10)):
        if counter==0:
            model.add(Dense(
                hp.Int('unit'+str(i),min_value=8,max_value=128,step=8),
                activation=hp.Choice('activation'+str(i),values=['relu','tanh','sigmoid']),
                input_dim=8
            ))
            model.add(Dropout(hp.Choice('dropout'+str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
        else:
            model.add(Dense(
                hp.Int('unit'+str(i),min_value=8,max_value=128,step=8),
                activation=hp.Choice('activation'+str(i),values=['relu','tanh','sigmoid'])
            ))
            model.add(Dropout(hp.Choice('dropout'+str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
        counter+=1
    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer=hp.Choice('optimizer',values=['adam','sgd','rmsprop','adadelta']),
                    loss='binary_crossentropy',
                    metrics=['accuracy'])
    return model
    
tunner4=kt.RandomSearch(build_model4,objective='val_accuracy',max_trials=3,directory='Mydir',project_name='allinone')
tunner4.search(xtrain,ytrain,epochs=5,validation_data=(xtest,ytest))


Trial 3 Complete [00h 00m 04s]
val_accuracy: 0.6428571343421936

Best val_accuracy So Far: 0.649350643157959
Total elapsed time: 00h 00m 15s


In [29]:
tunner4.get_best_hyperparameters()[0].values
model=tunner4.get_best_models(num_models=1)[0]
model.fit(xtrain,ytrain,epochs=100,initial_epoch=5,validation_data=(xtest,ytest))

c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 6/100


c:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.6450 - loss: 0.6640 - val_accuracy: 0.6429 - val_loss: 0.6541
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6580 - loss: 0.6491 - val_accuracy: 0.6429 - val_loss: 0.6436
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6547 - loss: 0.6312 - val_accuracy: 0.6429 - val_loss: 0.6310
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6564 - loss: 0.6313 - val_accuracy: 0.6429 - val_loss: 0.6133
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6498 - loss: 0.6265 - val_accuracy: 0.6429 - val_loss: 0.6034
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6531 - loss: 0.6277 - val_accuracy: 0.6429 - val_loss: 0.5990
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6629 - loss: 0.6148 - val_accuracy: 0.6429 - val_loss: 0.5876
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6498 - loss: 0.6207 - val_accuracy: 0.6494 - val_l